# 00 — Recolección y descripción del dataset

**Proyecto delimitado:** análisis territorial de alta no asistencia educativa en población con edades aproximadas de **sexto grado a bachillerato**.

La tabla censal no contiene el grado cursado ni edades simples. Por eso se conservan únicamente los grupos quinquenales **10 a 14 años** y **15 a 19 años**. Esta es una aproximación y puede incluir estudiantes con adelanto o extraedad, personas fuera del sistema educativo y edades que no coinciden exactamente con un grado.

In [1]:
from pathlib import Path
import pandas as pd

from src.proyecto_ml import (
    COLUMNAS_ORIGINALES, COLUMNAS_CONTEO, GRUPOS_ETARIOS_ALCANCE,
    cargar_tabla_original, filtrar_nivel_distrito, filtrar_sexto_bachillerato,
    localizar_raiz, verificar_blancos_como_cero,
)

RAIZ = localizar_raiz()
RAW = RAIZ / "data" / "raw"
TABLES = RAIZ / "outputs" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 30)

## Fuente, cobertura y licencia

- **Institución:** Banco Central de Reserva de El Salvador / Oficina Nacional de Estadística y Censos.
- **Operación estadística:** VII Censo de Población y VI de Vivienda 2024.
- **Tabla:** `TAB_EDU_1`, población de 3 años o más por territorio, grupo quinquenal, sexo y asistencia a educación formal.
- **Archivo utilizado:** `data/raw/TAB_EDU_1.xlsx`.
- **Fuente oficial:** portal del Censo 2024 del BCR.
- **Licencia:** el archivo es público, pero no se identificó dentro del archivo una licencia abierta estandarizada como CC BY, CC0 u ODbL. Se conserva atribución y el uso del proyecto es académico. Para reutilización comercial debe consultarse al BCR.

El proyecto no presenta los datos como registros individuales: cada fila es una combinación agregada de territorio y grupo etario.

In [2]:
metadata = pd.DataFrame([
    ["Institución", "Banco Central de Reserva de El Salvador / ONEC"],
    ["Operación", "VII Censo de Población y VI de Vivienda 2024"],
    ["Tabla", "TAB_EDU_1 — Características de educación"],
    ["Unidad original", "Territorio + grupo quinquenal de edad"],
    ["Alcance del proyecto", "Grupos 10–14 y 15–19 años"],
    ["Interpretación del alcance", "Proxy de sexto grado a bachillerato; no grado individual"],
    ["Uso", "Académico, con atribución a la fuente"],
    ["Licencia detectada", "No se identificó licencia abierta estandarizada en el archivo"],
], columns=["Elemento", "Descripción"])
metadata.to_csv(TABLES / "metadata_fuente_alcance.csv", index=False, encoding="utf-8-sig")
metadata

,Elemento,Descripción
0,Institución,Banco Central de Reserva de El Salvador / ONEC
1,Operación,VII Censo de Población y VI de Vivienda 2024
2,Tabla,TAB_EDU_1 — Características de educación
3,Unidad original,Territorio + grupo quinquenal de edad
4,Alcance del proyecto,Grupos 10–14 y 15–19 años
5,Interpretación del alcance,Proxy de sexto grado a bachillerato; no grado ...
6,Uso,"Académico, con atribución a la fuente"
7,Licencia detectada,No se identificó licencia abierta estandarizad...


In [3]:
df_original = cargar_tabla_original(RAW / "TAB_EDU_1.xlsx")
df_distrito = filtrar_nivel_distrito(df_original)
df_alcance = filtrar_sexto_bachillerato(df_distrito)

resumen_tamano = pd.DataFrame({
    "Etapa": ["Tabla cargada", "Nivel distrito sin totales", "Alcance 10–19 años"],
    "Filas": [len(df_original), len(df_distrito), len(df_alcance)],
    "Columnas": [df_original.shape[1], df_distrito.shape[1], df_alcance.shape[1]],
})
resumen_tamano.to_csv(TABLES / "tamano_dataset.csv", index=False, encoding="utf-8-sig")
resumen_tamano

,Etapa,Filas,Columnas
0,Tabla cargada,6735,14
1,Nivel distrito sin totales,5235,14
2,Alcance 10–19 años,522,14


In [4]:
print("Grupos etarios conservados:", sorted(df_alcance["grupo_etario"].unique()))
print("Departamentos:", df_alcance["departamento"].nunique())
print("Municipios:", df_alcance["municipio"].nunique())
print("Distritos:", df_alcance["distrito"].nunique())
df_alcance.head()

Grupos etarios conservados: ['10 a 14 años', '15 a 19 años']
Departamentos: 14
Municipios: 44
Distritos: 262


,departamento,municipio,distrito,grupo_etario,ha_asistido_hombres,ha_asistido_mujeres,ha_asistido_total,nunca_asistio_hombres,nunca_asistio_mujeres,nunca_asistio_total,no_sabe_hombres,no_sabe_mujeres,no_sabe_total,poblacion_total
0,01 - Ahuachapán,01- Ahuachapán Centro,01 - Ahuachapán,10 a 14 años,184.0,195.0,379.0,43.0,29.0,72.0,2.0,NaN,2.0,453.0
1,01 - Ahuachapán,01- Ahuachapán Centro,01 - Ahuachapán,15 a 19 años,1445.0,1566.0,3011.0,59.0,39.0,98.0,2.0,4.0,6.0,3115.0
2,01 - Ahuachapán,01- Ahuachapán Centro,02 - Apaneca,10 a 14 años,11.0,15.0,26.0,2.0,NaN,2.0,1.0,NaN,1.0,29.0
3,01 - Ahuachapán,01- Ahuachapán Centro,02 - Apaneca,15 a 19 años,118.0,116.0,234.0,3.0,3.0,6.0,3.0,NaN,3.0,243.0
4,01 - Ahuachapán,01- Ahuachapán Centro,04 - Concepción de Ataco,10 a 14 años,14.0,22.0,36.0,10.0,3.0,13.0,NaN,NaN,NaN,49.0


## Diccionario de variables originales

In [5]:
descripciones = {
    "departamento": "Departamento de residencia.",
    "municipio": "Municipio de residencia.",
    "distrito": "Distrito de residencia.",
    "grupo_etario": "Grupo quinquenal de edad.",
    "ha_asistido_hombres": "Hombres que actualmente asisten o asistieron alguna vez.",
    "ha_asistido_mujeres": "Mujeres que actualmente asisten o asistieron alguna vez.",
    "ha_asistido_total": "Total que actualmente asiste o asistió alguna vez.",
    "nunca_asistio_hombres": "Hombres que nunca asistieron a educación formal.",
    "nunca_asistio_mujeres": "Mujeres que nunca asistieron a educación formal.",
    "nunca_asistio_total": "Total que nunca asistió a educación formal.",
    "no_sabe_hombres": "Hombres clasificados como no sabe.",
    "no_sabe_mujeres": "Mujeres clasificadas como no sabe.",
    "no_sabe_total": "Total clasificado como no sabe.",
    "poblacion_total": "Población total de la fila agregada.",
}
diccionario = pd.DataFrame({
    "variable": COLUMNAS_ORIGINALES,
    "tipo": [str(df_alcance[c].dtype) for c in COLUMNAS_ORIGINALES],
    "descripcion": [descripciones[c] for c in COLUMNAS_ORIGINALES],
    "unidad": ["Categoría" if c in COLUMNAS_ORIGINALES[:4] else "Personas" for c in COLUMNAS_ORIGINALES],
})
diccionario.to_csv(TABLES / "diccionario_variables_originales.csv", index=False, encoding="utf-8-sig")
diccionario

,variable,tipo,descripcion,unidad
0,departamento,object,Departamento de residencia.,Categoría
1,municipio,object,Municipio de residencia.,Categoría
2,distrito,object,Distrito de residencia.,Categoría
3,grupo_etario,object,Grupo quinquenal de edad.,Categoría
4,ha_asistido_hombres,float64,Hombres que actualmente asisten o asistieron a...,Personas
5,ha_asistido_mujeres,float64,Mujeres que actualmente asisten o asistieron a...,Personas
6,ha_asistido_total,float64,Total que actualmente asiste o asistió alguna ...,Personas
7,nunca_asistio_hombres,float64,Hombres que nunca asistieron a educación formal.,Personas
8,nunca_asistio_mujeres,float64,Mujeres que nunca asistieron a educación formal.,Personas
9,nunca_asistio_total,float64,Total que nunca asistió a educación formal.,Personas


## Calidad inicial

Las celdas vacías de conteos se interpretan como cero únicamente después de comprobar las identidades aditivas del tabulado.

In [6]:
calidad = pd.DataFrame({
    "variable": COLUMNAS_ORIGINALES,
    "nulos": [int(df_alcance[c].isna().sum()) for c in COLUMNAS_ORIGINALES],
    "unicos": [int(df_alcance[c].nunique(dropna=True)) for c in COLUMNAS_ORIGINALES],
})
identidades = verificar_blancos_como_cero(df_alcance)
calidad.to_csv(TABLES / "calidad_inicial_columnas.csv", index=False, encoding="utf-8-sig")
print("Identidades máximas después de interpretar blancos como cero:", identidades)
calidad

Identidades máximas después de interpretar blancos como cero: {'error_max_ha_asistido': 0.0, 'error_max_nunca_asistio': 0.0, 'error_max_no_sabe': 0.0, 'error_max_poblacion': 0.0}


,variable,nulos,unicos
0,departamento,0,14
1,municipio,0,44
2,distrito,0,262
3,grupo_etario,0,2
4,ha_asistido_hombres,3,236
5,ha_asistido_mujeres,2,231
6,ha_asistido_total,1,305
7,nunca_asistio_hombres,76,47
8,nunca_asistio_mujeres,74,35
9,nunca_asistio_total,39,64


## Limitación de alcance

No es posible aislar exactamente a estudiantes de sexto grado a bachillerato porque la fuente:

1. agrupa edades en intervalos de cinco años;
2. no incluye grado cursado;
3. combina asistencia actual con asistencia alguna vez;
4. presenta información territorial agregada, no estudiantes individuales.

La delimitación usada es la aproximación más cercana disponible sin inventar registros.